In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
import os
import json

base_dir = Path("/home/[BLIND]/nlp/llm-as-tutor/result_analysis/bge/analysed")

In [ ]:
def comparison_between_cluster_factors(json_files: list, subset_key='subset_train_dev'):
    plt.figure(figsize=(10, 6))

    for file_path in json_files:
        if not file_path.exists():
            print(f"ERROR: {file_path}")
            exit(-1)
        else:
            x = json.loads(file_path.read_text())
            model_name = x['model_name']
            seed = x['seed']
            cluster_factor = x['cluster_factor']
            label_name = f"Seed={seed}"

            hist = np.array(x['embeddings_info'][subset_key]['cosine_hist'])
            bins = np.array(x['embeddings_info'][subset_key]['cosine_bins'])

            # calculo do centro p/ manter uma linha continua
            centers = (bins[:-1] + bins[1:]) / 2
            print(x['exp_name'], x['embeddings_info'][subset_key]['cosine_stats'])
            plt.plot(centers, hist, label=label_name, linewidth=2)
            plt.fill_between(centers, hist, alpha=0.2)
    #plt.title(f"Cosine Similarity Distribution - {model_name} | Cluster Factor={cluster_factor} | Subset={subset_key}", fontsize=14)
    plt.title(f"Cosine Similarity Distribution - {model_name} | Cluster Factor={cluster_factor}", fontsize=14)
    plt.xlabel("Cosine Similarity", fontsize=12)
    plt.ylabel("Pair frequency", fontsize=12)
    plt.xlim(-1.0, 1.0)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(loc='upper right')
    plt.tight_layout()

    # plt.save(f"{model_name}_{exp_name}_cosine_comparison.png", dpi=300)
    plt.savefig(f"cos_sim_dist_{model_name}_clust_factor={cluster_factor}.svg", dpi=3000)
    plt.show()
    return x['exp_name'], x['embeddings_info'][subset_key]['cosine_stats']


In [ ]:
bge_f1_json_files = [
    base_dir / "F1.0_S0.json",
    base_dir / "F1.0_S1.json",
    base_dir / "F1.0_S2.json",
]

bge_f2_json_files = [
    base_dir / "F2.0_S0.json",
    base_dir / "F2.0_S1.json",
    base_dir / "F2.0_S2.json",
]

bge_f3_json_files = [
    base_dir / "F3.0_S0.json",
    base_dir / "F3.0_S1.json",
    base_dir / "F3.0_S2.json",
]

In [ ]:
def analyze_clustering_metrics(json_files: list):
    cluster_stats = []
    
    for file_path in json_files:
        if not file_path.exists():
            print(f"[SKIP] Arquivo não encontrado: {file_path}")
            continue
            
        stats = json.loads(file_path.read_text())
        
        model_name = stats.get('model_name', 'Unknown')
        factor = stats.get('cluster_factor', 'Unknown')
        seed = stats.get('seed', 'Unknown')
        
        total_clusters = stats.get('num_labels', len(stats.get('silhouette_per_cluster', [])))
        global_silh = stats.get('silhouette_global', None)
        intra_sim = stats.get('intra_cluster_similarity_cos', None)   # <--
        inter_dist = stats.get('inter_cluster_distance', None)         # <--
        
        clusters_info = stats.get('silhouette_per_cluster', [])
        
        if clusters_info:
            silh_scores = [c['silhouette_score'] for c in clusters_info]
            sample_counts = [c['sample_count'] for c in clusters_info]
            
            max_silh = max(silh_scores)
            min_silh = min(silh_scores)
            mean_instances = sum(sample_counts) / len(sample_counts)
        else:
            max_silh = None
            min_silh = None
            mean_instances = None
            
        cluster_stats.append({
            'Model': model_name,
            'Factor': f"F{factor}",
            'Seed': f"S{seed}",
            'Total Clusters': total_clusters,
            'Mean Instances': mean_instances,
            'Global Silh': global_silh,
            'Max Silh': max_silh,
            'Min Silh': min_silh,
            'Intra Sim': intra_sim,     # <--
            'Inter Dist': inter_dist,   # <--
        })
        
    df_stats = pd.DataFrame(cluster_stats)
    
    format_dict = {
        'Mean Instances': lambda x: f"{x:.1f}" if pd.notnull(x) else "N/A",
        'Global Silh': lambda x: f"{x:.4f}" if pd.notnull(x) else "N/A",
        'Max Silh': lambda x: f"{x:.4f}" if pd.notnull(x) else "N/A",
        'Min Silh': lambda x: f"{x:.4f}" if pd.notnull(x) else "N/A",
        'Intra Sim': lambda x: f"{x:.4f}" if pd.notnull(x) else "N/A",   # <--
        'Inter Dist': lambda x: f"{x:.4f}" if pd.notnull(x) else "N/A",  # <--
    }
    
    print(df_stats.to_string(index=False, formatters=format_dict))
    return df_stats

In [ ]:
print("\n=== METRICAS DE CLUSTERING (BGE) ===")
df_scibert_metrics = analyze_clustering_metrics(
    bge_f1_json_files + bge_f2_json_files + bge_f3_json_files
)

## SUBSET TRAIN_DEV

In [ ]:
comparison_between_cluster_factors(json_files=bge_f1_json_files, subset_key='subset_train_dev')

In [ ]:
comparison_between_cluster_factors(json_files=bge_f2_json_files, subset_key='subset_train_dev')

In [ ]:
comparison_between_cluster_factors(json_files=bge_f3_json_files, subset_key='subset_train_dev')

## SUBSET ALL

In [ ]:
comparison_between_cluster_factors(json_files=bge_f1_json_files, subset_key='subset_all')

In [ ]:
comparison_between_cluster_factors(json_files=bge_f2_json_files, subset_key='subset_all')

In [ ]:
comparison_between_cluster_factors(json_files=bge_f3_json_files, subset_key='subset_all')

In [ ]:
bge_json_files = [
    base_dir / "F1.0_S0.json",
    base_dir / "F1.0_S1.json",
    base_dir / "F1.0_S2.json",

    base_dir / "F2.0_S0.json",
    base_dir / "F2.0_S1.json",
    base_dir / "F2.0_S2.json",

    base_dir / "F3.0_S0.json",
    base_dir / "F3.0_S1.json",
    base_dir / "F3.0_S2.json",
]

for file_path in bge_json_files:
    if not file_path.exists():
        print(f"ERROR: {file_path}")
        exit(-1)
    else:
        x = json.loads(file_path.read_text())
        stats = x['embeddings_info']['subset_train_dev']['cosine_stats']
        
        print(x['exp_name'], stats['mean'], stats['std'],  x['embeddings_info']['subset_train_dev']['95th_percentile'])


In [ ]:
bge_f1 = [
    base_dir / "F1.0_S0.json",
    base_dir / "F1.0_S1.json",
    base_dir / "F1.0_S2.json",
]

bge_f2 = [
    base_dir / "F2.0_S0.json",
    base_dir / "F2.0_S1.json",
    base_dir / "F2.0_S2.json",
]

bge_f3 = [
    base_dir / "F3.0_S0.json",
    base_dir / "F3.0_S1.json",
    base_dir / "F3.0_S2.json",
]

def eigenvalues(json_files, subset_key):
    offsets = [(20, 20), (55, 0), (20, -20)]
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    plt.figure(figsize=(12, 8))

    for i, file_path in enumerate(json_files):
        if not file_path.exists():
            print(f"ERROR: {file_path}")
            exit(-1)
        else:
            stats = json.loads(file_path.read_text())
            model_name = stats['model_name']
            cluster_factor = stats['cluster_factor']
            seed = stats['seed']
            label_name = f"Seed={seed}"
            eigenvalues = stats['embeddings_info'][subset_key]['eigenvalues']

            plot_range = range(1, len(eigenvalues) + 1)
            total_var = sum(eigenvalues)
            variancia_ratio = [ev / total_var for ev in eigenvalues]
            
            highest_ratio = variancia_ratio[0]
            
            plt.plot(plot_range, variancia_ratio, linewidth=2, label=label_name, color=colors[i])
    
            plt.annotate(
                    f"{highest_ratio * 100:.2f}%",         
                    xy=(1, highest_ratio),                 
                    xytext=offsets[i],                    
                    textcoords='offset points',           
                    ha='left', va='center',               
                    color=colors[i],                      
                    arrowprops=dict(
                        arrowstyle='-|>',                 
                        color=colors[i],                  
                        connectionstyle='arc3,rad=0.1'    
                    )
                )

    plt.title(f'{model_name} | Cluster Factor={cluster_factor} | Subset={subset_key} - Isotropy analysis by Eigen Values')
    plt.xlabel('Principal Component')
    plt.ylabel('Explained variance proportion')
    plt.legend(loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.show()

In [ ]:
eigenvalues(bge_f1, subset_key='subset_train_dev')

In [ ]:
eigenvalues(bge_f2, subset_key='subset_train_dev')

In [ ]:
eigenvalues(bge_f3, subset_key='subset_train_dev')

In [ ]:
def silhouette_scores_per_clusters(json_files):
    all_cluster_data = []
    
    for i, file_path in enumerate(json_files):
        if not file_path.exists():
            print(f"[SKIP] Erro, arquivo não encontrado: {file_path}")
            continue 
            
        stats = json.loads(file_path.read_text())
        cluster_factor = stats['cluster_factor']
        seed = stats['seed']
        clusters = stats['silhouette_per_cluster']
        num_labels_json = stats['num_labels']

        qtd_clusters = len(clusters)
        print(f"[{file_path.name}] Factor: {cluster_factor} | Seed: {seed} | Clusters Encontrados: {qtd_clusters} | Num_Labels: {num_labels_json}")

        for c in clusters:
            all_cluster_data.append({
                'Factor': f"F{cluster_factor}",
                'Seed': f"S{seed}",
                'Experiment': f"F{cluster_factor}_S{seed}",
                'Cluster_ID': c['cluster_id'],
                'Silhouette': c['silhouette_score']
            })

    df_plot = pd.DataFrame(all_cluster_data)
    
    print("\n=== STATS SILHOUETTE SCORE PER EXPERIMENT ===")
    summary = df_plot.groupby('Experiment')['Silhouette'].describe()[['mean', '50%', 'min', 'max']]
    summary = summary.rename(columns={'50%': 'mediana'})
    print(summary.to_string())

    # boxplot e pontos
    plt.figure(figsize=(10, 6))

    sns.boxplot(x='Experiment', y='Silhouette', data=df_plot, hue='Experiment', palette='Set2', showfliers=False, legend=False)

    sns.stripplot(x='Experiment', y='Silhouette', data=df_plot, color=".2", alpha=0.6, size=4, jitter=True)

    plt.title(f"Global Silhouette Distribution per Experiment - {cluster_factor}", fontsize=14)
    plt.xlabel("Experiment (Factor + Seed)", fontsize=12)
    plt.ylabel("Cluster Silhouette Score", fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()

    plt.savefig("silhouette_all_clusters.png", dpi=300)
    plt.show()

In [ ]:
silhouette_scores_per_clusters(bge_f1)

In [ ]:
silhouette_scores_per_clusters(bge_f2)

In [ ]:
silhouette_scores_per_clusters(bge_f3)

In [ ]:
SILHOUETTE_THRESHOLD = 0.20

def best_results(json_files: list, t: float = SILHOUETTE_THRESHOLD):
    best_clusters = []

    for file_path in json_files:
        if not file_path.exists():
            print(f"[SKIP] Arquivo não encontrado: {file_path}")
            continue
        else:
            stats = json.loads(file_path.read_text())
            model_name = stats['model_name']
            cluster_factor = stats['cluster_factor']
            seed = stats['seed']
            experiment_name = f"{model_name}_F{cluster_factor}_S{seed}"
            clusters = stats['silhouette_per_cluster']
            for c in clusters:
                score = c['silhouette_score']

                if score >= t:
                    best_clusters.append({
                        'Experiment': experiment_name,
                        'Factor': f"F{cluster_factor}",
                        'Seed': seed,
                        'Cluster_ID': c['cluster_id'],
                        'Silhouette': score,
                        'Size': c['sample_count']
                    })

    df_ranked = pd.DataFrame(best_clusters)
    
    if not df_ranked.empty:
        df_ranked = df_ranked.sort_values(by=['Factor', 'Silhouette'], ascending=[True, False])
        df_ranked = df_ranked.reset_index(drop=True)
    
    return df_ranked

In [ ]:
df_f1_top_clusters = best_results(bge_f1_json_files)

print(f"TOTAL EXTRACTED CLUSTERS = {SILHOUETTE_THRESHOLD}: {len(df_f1_top_clusters)}")
print(df_f1_top_clusters.head(len(df_f1_top_clusters)))

In [ ]:
df_f2_top_clusters = best_results(bge_f2_json_files)

print(f"TOTAL EXTRACTED CLUSTERS = {SILHOUETTE_THRESHOLD}: {len(df_f2_top_clusters)}")
print(df_f2_top_clusters.head(len(df_f2_top_clusters)))

In [ ]:
df_f3_top_clusters = best_results(bge_f3_json_files)

print(f"TOTAL EXTRACTED CLUSTERS = {SILHOUETTE_THRESHOLD}: {len(df_f3_top_clusters)}")
print(df_f3_top_clusters.head(len(df_f3_top_clusters)))

SPECTER2 com CLUSTER FACTOR = 2 tem, aparentemente mais clusters com silhouette scores.

In [ ]:
def comparison_results_inter(json_files: list):
    for file_path in json_files:
        if not file_path.exists():
            print(f"[SKIP] Arquivo não encontrado: {file_path}")
            continue
        else:
            stats = json.loads(file_path.read_text())
            model_name = stats['model_name']
            cluster_factor = stats['cluster_factor']
            seed = stats['seed']
            # experiment_name = f"{model_name}_F{cluster_factor}_S{seed}"
            global_silhouette = stats['silhouette_global']
            intra_sim = stats['intra_cluster_similarity_cos']
            inter_dist =  stats['inter_cluster_distance']

            print(f"\n---Factor={cluster_factor}; seed={seed}--")
            print(f"SILHOUETTE GLOBAL=", global_silhouette)
            print(f"INTRA SIMILARITY=", intra_sim)
            print(f"INTER DISTANCE: Factor={cluster_factor}; seed={seed} -->", inter_dist)
            print(f"------------------------------------------")

In [ ]:
comparison_results_inter(bge_f1_json_files)

In [ ]:
comparison_results_inter(bge_f2_json_files)

In [ ]:
comparison_results_inter(bge_f3_json_files)

#  analise morfosintatica

In [ ]:
import json
from pathlib import Path

def is_space_healthy(run_folder: Path) -> bool:
    json_path = run_folder.parent / f"{run_folder.name}.json"
    
    if not json_path.exists():
        print(f"  [ERRO] JSON não encontrado para ler anisotropia: {json_path}")
        return False
        
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            stats = json.load(f)
    except Exception as e:
        print(f"  [ERRO] Falha ao abrir o JSON {json_path}: {e}")
        return False
    
    try:
        cosine_data = stats['embeddings_info']['subset_train_dev']['cosine_stats']
        mu = float(cosine_data['mean'])
        p75 = float(cosine_data['p75'])
                
    except Exception as e:
        print(f"  [ERRO] Chaves não encontradas no JSON {json_path.name}. Detalhe: {e}")
        return False

    if abs(mu) <= 0.15 and p75 >= 0.20:
        return True
    else:
        print(f"  [REJEITADO] {run_folder.name}: mu={mu:.3f}, p75={p75:.3f} (Fora do limiar seguro)")
        return False

In [ ]:
def classify_sentence_type(doc):
    has_modal = False
    has_neg = False
    has_negative_adj = False
    has_positive_adj = False

    for token in doc:
        if token.tag_ == "MD":
            has_modal = True
        
        if token.dep_ == "neg":
            has_neg = True
        
        if token.pos_ == "ADJ":
            lemma = token.lemma_.lower()
            
            if lemma in ["good", "effective", "promising", "strong"]:
                has_positive_adj = True
            
            if lemma in ["unclear", "incorrect", "weak", "limited"]:
                has_negative_adj = True

    if has_modal:
        return "SUGGESTION"
    
    if has_neg or has_negative_adj:
        return "CRITICISM"
    
    if has_positive_adj:
        return "PRAISE"
    
    return "NEUTRAL"

In [ ]:
from collections import Counter
from tqdm import tqdm
import spacy
spacy.prefer_gpu()
nlp = spacy.load("en_core_web_trf")

special_tokens = ["[FIG]", "[TAB]", "[SEC]", "[EQ]", "[REF]", "[TECH]"]
for token in special_tokens:
    nlp.tokenizer.add_special_case(token, [{"ORTH": token}])

deps_to_keep = {"neg", "prt", "amod", "compound", "xcomp", "ccomp", "dobj", "pobj", "csubj", "csubjpass", "conj", "nsubj", "nsubjpass"}
custom_stop_nouns = {"something", "thing", "it", "author", "authors", 
                     "et", "al", "work", "study", "result", "section", "like", "idea", "loss",
                     "layer", "training", "decoder", "network", "bit", "clustering", "page", "term"}

custom_stop_verbs = {"be", "have", "do", "make", "seem", "look", "like"}


def extract_action_v3(doc):
    actions = []
    seen = set()

    # 1. Encontrar a Raiz
    roots = [t for t in doc if t.dep_ == "ROOT"]
    if not roots: return []
    root = roots[0]

    content_clauses = [c for c in root.children if c.dep_ in ["ccomp", "csubj", "csubjpass", "acl"] and c.pos_ == "VERB"]
    if content_clauses:
        base_node = content_clauses[0] # A seleção cai no verbo do complemento
    else:
        base_node = root

    acomps = [c for c in base_node.children if c.dep_ == "acomp"]
    if acomps:
        base_node = acomps[0] 

    if base_node.pos_ not in ["VERB", "ADJ"]:
        return []

    xcomps = [c for c in base_node.children if c.dep_ == "xcomp" and c.pos_ == "VERB"]
    if xcomps:
        base_node = xcomps[0]

    nodes_to_process = [base_node]
    nodes_to_process.extend([c for c in base_node.children if c.dep_ == "conj" and c.pos_ in ["VERB", "ADJ"]])

    for node in nodes_to_process:
        modifiers = []
        for c in node.children:
            if c.dep_ in ["prt", "neg"]:
                modifiers.append(c)
            # Negação via advmod
            if c.dep_ == "advmod":
                modifiers.extend([cc for cc in c.children if cc.dep_ == "neg"])


        current_head = node.head
        while current_head and current_head != current_head.head:
            modifiers.extend([c for c in current_head.children if c.dep_ == "neg"])
            if current_head.dep_ == "ROOT":
                break
            current_head = current_head.head
        if node.lemma_.lower() in custom_stop_verbs:
            continue
        
        modifiers = sorted(list(set(modifiers)), key=lambda x: x.i)
        
        mod_text = " ".join([m.lemma_ for m in modifiers])
        label = f"{mod_text} {node.lemma_}".strip()

        if label not in seen:
            actions.append((node, label))
            seen.add(label)

    return actions


In [ ]:

def expand_noun_v3(token):
    parts = []
    for c in token.children:
        if c.dep_ in ["compound", "prt"]:
            if c.lemma_.lower() not in custom_stop_nouns:
                parts.append(c)
            
    parts.append(token)
    parts = sorted(parts, key=lambda x: x.i)
    return " ".join([p.lemma_ for p in parts])

def extract_object_v3(action_node):
    spcy_obj = ["NOUN", "PROPN"] 

    candidates = [
        t for t in action_node.children
        if t.dep_ in ["dobj", "pobj", "nsubj", "nsubjpass"]
        and t.pos_ in spcy_obj
    ]
    objects = []
    for t in candidates:
        lemma_core = t.lemma_.lower()
        
        if lemma_core not in custom_stop_nouns:
            objects.append(expand_noun_v3(t))
            
    return objects

In [ ]:

def generate_cluster_label_v3(sentences):
    pair_counter = Counter()
    extra_words_counter = Counter()
    type_counter = Counter()
    
    sents_with_action = 0

    for doc in nlp.pipe(sentences, batch_size=64):
        sent_type = classify_sentence_type(doc)
        type_counter[sent_type] += 1

        actions = extract_action_v3(doc)
        
        if actions:
            sents_with_action += 1

        seen_pairs = set()
        seen_words = set()

        for action_node, action_str in actions:
            objs = extract_object_v3(action_node)

            for obj_str in objs:
                pair = f"{action_str} {obj_str}".lower()
                if pair not in seen_pairs:
                    pair_counter[pair] += 1
                    seen_pairs.add(pair)
                    
        for token in doc:
            if token.pos_ in ["NOUN", "PROPN"]:
                is_stop = token.is_stop
                has_vital_dep = token.dep_ in deps_to_keep
                lemma_core = token.lemma_.lower()
                
                if not (is_stop and not has_vital_dep) and lemma_core not in custom_stop_nouns:
                    word = expand_noun_v3(token).lower() 
                    if word not in seen_words:
                        extra_words_counter[word] += 1
                        seen_words.add(word)

    total_sents = len(sentences)
    if type_counter:
        dominant_type, freq = type_counter.most_common(1)[0]
        type_ratio = freq / total_sents
    else:
        dominant_type, type_ratio = "NEUTRAL", 0.0

    if type_ratio >= 0.4:
        cluster_type = dominant_type
    elif type_ratio >= 0.25:
        cluster_type = "WEAK_" + dominant_type
    else:
        cluster_type = "MIXED"

    
    top_pairs = pair_counter.most_common(1)

    if cluster_type == "SUGGESTION" and top_pairs:
        label_main = top_pairs[0][0]
        main_words = set(label_main.split())
        strategy_tag = "[SUGGESTION]"

    elif cluster_type == "CRITICISM":
        top_words = [w for w, _ in extra_words_counter.most_common(3)]
        label_main = " ".join(top_words)
        main_words = set(label_main.split())
        strategy_tag = "[CRITICISM]"

    elif cluster_type == "PRAISE":
        label_main = "positive evaluation"
        main_words = set(label_main.split())
        strategy_tag = "[PRAISE]"

    else:
        if top_pairs:
            freq_top_pair = top_pairs[0][1]
            action_strength = freq_top_pair / total_sents
        else:
            action_strength = 0.0

        percent_of_cluster = 0.09

        if top_pairs and action_strength >= percent_of_cluster and freq_top_pair >= 2:
            label_main = top_pairs[0][0]
            main_words = set(label_main.split())
            strategy_tag = "[ACTION]"
        else:
            top_words = [w for w, _ in extra_words_counter.most_common(3)]
            label_main = " ".join(top_words)
            main_words = set(label_main.split())
            strategy_tag = "[TOPIC]"
    

    extra_label = []
    for word, count in extra_words_counter.most_common(10):
        if word not in main_words and count > 1:
            is_redundant = any(word in m or m in word for m in main_words)
            if not is_redundant:
                extra_label.append(word)
        if len(extra_label) == 2:
            break

    if extra_label:
        final_label = f"{strategy_tag} [{label_main}] + {' '.join(extra_label)}"
    else:
        final_label = f"{strategy_tag} [{label_main}]"

    return {
        "label": final_label,
        "type": cluster_type
    }

In [ ]:
def get_top_clusters_hierarchical(model_name: str, base_dir_path: str, min_silhouette: float = 0.20):
    base_dir = Path(base_dir_path)
    selected_clusters = []
    
    
    for run_folder in base_dir.iterdir():
        if not run_folder.is_dir() or "F" not in run_folder.name:
            continue
            
        parts = run_folder.name.split('_')
        factor = parts[0].replace('F', '')
        seed = parts[1].replace('S', '')
        
        if not is_space_healthy(run_folder):
            print(f"  [REJEITADO] {run_folder.name}: Falhou no filtro de saúde geométrica (Anisotropia).")
            continue
            
        print(f"  [APROVADO] {run_folder.name}: Espaço saudável. Garimpando clusters...")
        
        csv_path = run_folder / "silhouette_per_cluster.csv"
        
        if csv_path.exists():
            df_sil = pd.read_csv(csv_path)
            
            good_clusters = df_sil[df_sil['silhouette_score'] >= min_silhouette]
            
            for _, row in good_clusters.iterrows():
                selected_clusters.append({
                    "Model": model_name,
                    "Factor": factor,
                    "Seed": seed,
                    "Folder": run_folder.name,
                    "Cluster_ID": int(row["cluster_id"]),
                    "Silhouette": row["silhouette_score"],
                    "Size": int(row.get("sample_count", 0))
                })
                
    df_results = pd.DataFrame(selected_clusters)
    if not df_results.empty:
        df_results = df_results.sort_values(by="Silhouette", ascending=False).reset_index(drop=True)
        
    print(f"\nTotal de Clusters Sobreviventes (Passaram nos 2 Filtros): {len(df_results)}")
    return df_results

In [ ]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm

base_dir = Path("/home/[BLIND]/nlp/llm-as-tutor/result_analysis/bge/analysed")

def labelling_by_factor(df_top_clusters):
    results_list = []

    for idx, row in tqdm(df_top_clusters.iterrows(), total=len(df_top_clusters), desc=""):
        
        folder_name = f"{row['Factor']}_S{row['Seed']}"
        cluster_id = int(row['Cluster_ID'])
        
        csv_path = base_dir / folder_name / "clusters_complete_Xtest.csv"
        
        if not csv_path.exists():
            print(f"[SKIP] CSV não encontrado: {csv_path}")
            continue
            
        df_csv = pd.read_csv(csv_path)
        
        df_cluster = df_csv[df_csv['cluster_id'] == cluster_id]
        cluster_texts = df_cluster['text'].tolist()
        
        if not cluster_texts:
            label = "EMPTY_CLUSTER"
            top_samples = []
        else:
            result = generate_cluster_label_v3(cluster_texts)
            print(type(result))
            label = result["label"]
            cluster_type = result["type"]
            
            if 'distance_to_centroid' in df_cluster.columns:
                top_samples = df_cluster.sort_values(by='distance_to_centroid', ascending=True)['text'].head(5).tolist()
            else:
                top_samples = cluster_texts[:5] 
        
        results_list.append({
            'Folder': folder_name,
            'Cluster_ID': cluster_id,
            'Silhouette': row['Silhouette'],
            'Size': len(cluster_texts),
            'Morph_Label': label,
            'Cluster_Type': cluster_type,
            'Sample_Texts': top_samples 
        })

    df_labeled_clusters = pd.DataFrame(results_list)

    for idx, row in df_labeled_clusters.iterrows():
        print("=" * 80)
        print(f"RUN: {row['Folder']} | CLUSTER ID: {row['Cluster_ID']} | SILHOUETTE: {row['Silhouette']:.3f} | SIZE: {row['Size']}")
        print(f"TYPE: {row['Cluster_Type']}")
        print(f"LABEL: {row['Morph_Label']}")
        print("- 5 Exemplos Mais Representativos (Mais próximos ao centroide):")
        for ex in row['Sample_Texts']:
            print(f"  • {ex}")
        print("\n")

In [ ]:
def run_cherry_picked_labeling(df_cherry_picked, base_dir_path: Path):
    results_list = []

    for idx, row in tqdm(df_cherry_picked.iterrows(), total=len(df_cherry_picked), desc="Processando Labels"):
        
        folder_name = row['Folder']
        cluster_id = row['Cluster_ID']
        
        csv_path = base_dir_path / folder_name / "clusters_complete_Xtest.csv"
        
        if not csv_path.exists():
            print(f"\n[SKIP] CSV não encontrado: {csv_path}")
            continue
            
        df_csv = pd.read_csv(csv_path)
        df_cluster = df_csv[df_csv['cluster_id'] == cluster_id]
        cluster_texts = df_cluster['text'].tolist()
        
        if not cluster_texts:
            continue
            
        result = generate_cluster_label_v3(cluster_texts)
        
        if 'distance_to_centroid' in df_cluster.columns:
            top_samples = df_cluster.sort_values(by='distance_to_centroid', ascending=True)['text'].head(5).tolist()
        else:
            top_samples = cluster_texts[:5] 
        
        results_list.append({
            'Folder': folder_name,
            'Factor': row['Factor'],
            'Seed': row['Seed'],
            'Cluster_ID': cluster_id,
            'Silhouette': row['Silhouette'],
            'Size': row['Size'],
            'Morph_Label': result["label"],
            'Cluster_Type': result["type"],
            'Sample_Texts': top_samples 
        })

    print("\n\n" + "="*80)
    print("                RESULTADOS DO LABELING AUTOMÁTICO")
    print("="*80)
    
    for row in results_list:
        print(f"RUN: {row['Folder']} | CLUSTER ID: {row['Cluster_ID']} | SILHOUETTE: {row['Silhouette']:.3f} | SIZE: {row['Size']}")
        print(f"TYPE: {row['Cluster_Type']}")
        print(f"LABEL: {row['Morph_Label']}")
        print("- 5 Exemplos Mais Representativos (Top Centroid):")
        for ex in row['Sample_Texts']:
            print(f"  • {ex}")
        print("-" * 80)

In [ ]:
THRESHOLD = 0.20

df_cherry_picked = get_top_clusters_hierarchical("bge", base_dir, THRESHOLD)

print("\nTop 5 Clusters Encontrados (Hierarquia Dupla):")
if not df_cherry_picked.empty:
    print(df_cherry_picked.head(5))

In [ ]:
if not df_cherry_picked.empty:
    run_cherry_picked_labeling(df_cherry_picked, base_dir)
else:
    print(f"Nenhum cluster superou o threshold de {THRESHOLD}.")
